# 03 — Webapp: Plot Boundary Detector (Gradio, Colab)

Launches a small web UI from Colab. **Pick a tile from the dropdown → see the detected plot boundaries on the aerial photo.** No code needed to use it.

Prereqs:
- `02_train_colab.ipynb` has produced `MyDrive/aigeolab_train/checkpoints/best.pt`.
- `01_prep_colab.ipynb` has produced `MyDrive/aigeolab_train/{tiles, manifest.csv}`.

**Runtime: GPU (T4 enough).** Run All. The last cell prints a `https://*.gradio.live` URL — open it on any device.

In [ ]:
# --- Cell 1: install deps + mount Drive + clone repo ---
!pip install -q gradio rasterio segmentation-models-pytorch albumentations opencv-python-headless pyyaml pyshp

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_URL = 'https://github.com/tahmid013/AIGEOLAB_OFFICE.git'
REPO_DIR = '/content/AIGEOLAB_OFFICE'
if os.path.isdir(REPO_DIR):
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True).stdout)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))


In [ ]:
# --- Cell 2: load model checkpoint + manifest ---
import yaml, csv
import torch, numpy as np, rasterio, cv2
from pathlib import Path
from inference import load_model, predict_heatmap, extract_polygons, draw_polygons, polygon_stats

with open('config.yaml') as f: CFG = yaml.safe_load(f)
STAGE = Path(CFG['paths']['colab']['staging_root'])
CKPT  = STAGE / 'checkpoints' / 'best.pt'
assert CKPT.exists(), f'Checkpoint not found: {CKPT}. Run 02_train_colab.ipynb first.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, model_cfg, meta = load_model(CKPT, device=DEVICE)
PATCH = model_cfg['dataset']['patch_size']
STRIDE = PATCH // 2
print(f'Loaded model from {CKPT}')
print(f'  trained for {meta["epoch"]} epochs, val_iou={meta["val_iou"]:.4f}')
print(f'  patch={PATCH} stride={STRIDE} device={DEVICE}')

# Build the dropdown options from manifest.csv
manifest = []
with open(STAGE / 'manifest.csv') as fh:
    for r in csv.DictReader(fh):
        manifest.append(r)

tile_options = []  # list of (display_label, tif_relpath)
for r in manifest:
    mouzas = r['mouza_label_stems'].replace('|', ', ').replace('03.251222__', '').replace('04.251226__', '')
    label = f"Tile {r['tile_x_km']}-{r['tile_y_km']}  ·  covers: {mouzas}"
    tile_options.append((label, r['tile_tif_relpath']))
print(f'{len(tile_options)} tiles available in the dropdown')


In [ ]:
# --- Cell 3: inference helpers wrapped for the app ---
PIXEL_SIZE_M = 0.1     # 10 cm per pixel (the imagery's GSD)
CROP_SIZE    = 2048    # 2048x2048 = ~205m x 205m, comfortable to view
DEFAULT_THRESHOLD = 0.4
DEFAULT_MIN_AREA  = 400

def load_tile_crop(tif_relpath, top_frac=0.5, left_frac=0.5, size=CROP_SIZE):
    '''Read a CROP_SIZE x CROP_SIZE crop from a tile. top_frac/left_frac in [0,1] = where in the tile.'''
    tif_path = STAGE / tif_relpath
    with rasterio.open(tif_path) as ds:
        H, W = ds.height, ds.width
        max_top = max(0, H - size); max_left = max(0, W - size)
        top  = int(max_top  * top_frac)
        left = int(max_left * left_frac)
        win = rasterio.windows.Window(left, top, min(size, W), min(size, H))
        arr = ds.read([1, 2, 3], window=win)
    return np.transpose(arr, (1, 2, 0)).astype(np.uint8)  # (H, W, 3)

def analyse(tile_relpath, top_frac, left_frac, threshold, min_area):
    '''Full pipeline: crop -> predict -> vectorise -> draw + summarise.'''
    rgb = load_tile_crop(tile_relpath, top_frac=top_frac, left_frac=left_frac)
    prob = predict_heatmap(model, rgb, patch=PATCH, stride=STRIDE, device=DEVICE)
    polys = extract_polygons(prob, threshold=threshold, min_area_px=int(min_area))
    overlay = draw_polygons(rgb, polys, color=(255, 215, 0), thickness=3, fill_alpha=0.10)
    stats = polygon_stats(polys, pixel_size_m=PIXEL_SIZE_M)
    # also show the raw heatmap as a coloured layer
    heat = (prob * 255).astype(np.uint8)
    heat_rgb = cv2.applyColorMap(heat, cv2.COLORMAP_VIRIDIS)[:, :, ::-1]  # BGR->RGB
    heat_blend = (rgb.astype(np.float32) * 0.45 + heat_rgb.astype(np.float32) * 0.55).clip(0, 255).astype(np.uint8)

    md = f'''### Detection summary

- **Plots detected:** {stats["n_plots"]}
- **Total area:** {stats["total_area_m2"]:,.0f} m²  ({stats["total_area_m2"]/10000:.2f} hectares)
- **Average plot size:** {stats["mean_area_m2"]:,.0f} m² ({stats["mean_area_m2"]/10000:.3f} ha)
- **Median plot size:** {stats["median_area_m2"]:,.0f} m²
- **Photo scale:** 10 cm per pixel  ·  {CROP_SIZE//10} m × {CROP_SIZE//10} m visible area

_Boundaries shown where the model is more than **{int(threshold*100)}%** confident. Plots smaller than **{int(min_area * PIXEL_SIZE_M ** 2):,} m²** are filtered as noise._
'''
    return rgb, overlay, heat_blend, md

print('Inference helpers ready.')


In [ ]:
# --- Cell 4: Gradio UI ---
import gradio as gr

CSS = '''
.gradio-container {max-width: 1400px !important}
h1 {font-size: 1.7rem !important}
h3 {margin-top: 0.5em !important}
'''

with gr.Blocks(title='AIGEOLAB · Plot Boundary Detector', css=CSS, theme=gr.themes.Soft()) as app:
    gr.Markdown('# Plot Boundary Detector — Bangladesh')
    gr.Markdown(
        'Pick an aerial photo from the dropdown below. The model traces every plot boundary it can see. '
        'The yellow outlines on the right are the detected plots. Adjust the sliders if you want more or fewer plots.'
    )

    with gr.Row():
        with gr.Column(scale=2):
            tile_dd = gr.Dropdown(
                choices=tile_options,
                label='1.  Select an aerial photo',
                value=tile_options[0][1] if tile_options else None,
                interactive=True,
            )
        with gr.Column(scale=1):
            top_slider  = gr.Slider(0, 1, value=0.5, step=0.1, label='Vertical crop position (top=0, bottom=1)')
            left_slider = gr.Slider(0, 1, value=0.5, step=0.1, label='Horizontal crop position (left=0, right=1)')

    with gr.Accordion('Advanced — detection sensitivity', open=False):
        thr_slider  = gr.Slider(0.1, 0.8, value=DEFAULT_THRESHOLD, step=0.05, label='Boundary confidence threshold (lower = more boundaries)')
        area_slider = gr.Slider(50, 5000, value=DEFAULT_MIN_AREA, step=50, label='Minimum plot area (pixels) — filters noise')

    btn = gr.Button('2.  Analyze this photo', variant='primary', size='lg')

    summary = gr.Markdown()

    with gr.Row():
        img_orig  = gr.Image(label='Aerial photo', show_label=True, height=500)
        img_out   = gr.Image(label='Detected plot boundaries (yellow)', show_label=True, height=500)
    with gr.Accordion('Show raw model heatmap', open=False):
        img_heat  = gr.Image(label='Boundary probability heatmap (bright = model is more sure this is an edge)', show_label=True, height=500)

    btn.click(
        analyse,
        inputs=[tile_dd, top_slider, left_slider, thr_slider, area_slider],
        outputs=[img_orig, img_out, img_heat, summary],
    )

    gr.Markdown(
        '---\n'
        f'<small>Model: U-Net (ResNet-34) trained for {meta["epoch"]} epochs · val IoU (masked) {meta["val_iou"]:.3f}'
        f' · trained on {len(manifest)} tiles · imagery resolution 10 cm/px</small>'
    )


In [ ]:
# --- Cell 5: launch the app (prints a public gradio.live URL) ---
app.queue(default_concurrency_limit=1).launch(share=True, inline=False, debug=False)
